# Interference Intensifiers

Having established the paradigm, what encoding-phase factors determine interference intensity?
Three candidates arise from CMR's architecture:

1. **Interference MCF encoding strength** — how strongly competitors are bound into M\_CF.
Stronger associations mean higher retrieval support for competitors, diverting probability mass away from film items during competitive recall.

2. **Interference encoding drift rate** — how much temporal context drifts during the interference phase.
Low drift keeps competitors in film-adjacent context; high drift pushes them into a distant region.

3. **Competitor count** — the sheer number of interfering items.
More competitors mean more total competition, but with a caveat: later competitors accumulate more contextual drift from the film region, so each additional competitor produces diminishing returns.

Each is swept in isolation, holding the other two at their default values.

In [ ]:
import warnings
from pathlib import Path

import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from jax import random, jit, lax, vmap

from jaxcmr.analyses.spc import fixed_pres_spc
from jaxcmr.helpers import find_project_root, generate_trial_mask, load_data
from jaxcmr.selective_interference import (
    Paradigm,
    compute_n_presented,
    prepare_sweep,
    run_sweep,
    sweep_rngs,
    film_recalled_stats,
    load_or_fit_params,
    make_factory,
    standard_remap,
    interference_extended_remap,
    make_extended_interference,
    plot_interference_spc,
    plot_summary_dv,
    light_to_dark_colors,
    add_filler_boundary,
)
from jaxcmr.math import normalize_magnitude

warnings.filterwarnings("ignore")

In [ ]:
# --- paradigm geometry ---
N_FILM = 16
N_BREAK = 16
N_INTERFERENCE = 16
N_INTERFERENCE_MAX = 32
N_FILLER = 16
EXPERIMENT_COUNT = 100
SHOW_BREAK_IN_SPC = True
SHOW_FILLERS_IN_SPC = True

# --- default scale factors ---
DEFAULT_BREAK_DRIFT_SCALE = 1.0
DEFAULT_BREAK_MCF_SCALE = 1.0
DEFAULT_REMINDER_START_DRIFT_SCALE = 4.0
DEFAULT_REMINDER_DRIFT_SCALE = 0.3
DEFAULT_FILLER_DRIFT_SCALE = 1.0
DEFAULT_FILLER_MCF_SCALE = 1.0

# --- fitted parameter scales ---
PARAM_SCALES = {
    "stop_probability_scale": 0.57,
}

# --- fitting ---
SEED = 0
DATA_TAG = "HealeyKahana2014"
DATA_PATH = "data/HealeyKahana2014.h5"
TRIAL_QUERY = "data['listtype'] == -1"
MODEL_NAME = "WeirdCMRPosStop"
BEST_OF = 1
RUN_TAG = f"best_of_{BEST_OF}"
REDO_FITS = False

In [ ]:
# --- derived objects ---
paradigm = Paradigm(
    n_film=N_FILM, n_break=N_BREAK,
    n_interference=N_INTERFERENCE, n_interference_max=N_INTERFERENCE_MAX,
    n_filler=N_FILLER,
    experiment_count=EXPERIMENT_COUNT,
)

N_BREAK_SHOWN = N_BREAK if SHOW_BREAK_IN_SPC else 0
STD_PRESENTED = compute_n_presented(
    paradigm, show_break=SHOW_BREAK_IN_SPC, show_fillers=SHOW_FILLERS_IN_SPC,
)

project_root = Path(find_project_root())
fit_dir = Path("results/fits")
fit_dir.mkdir(parents=True, exist_ok=True)
fit_path = fit_dir / f"{DATA_TAG}_{MODEL_NAME}_{RUN_TAG}.json"

data = load_data(str(project_root / DATA_PATH))
trial_mask = generate_trial_mask(data, TRIAL_QUERY)
factory = make_factory()
rng = random.PRNGKey(SEED)

params, n_subjects = load_or_fit_params(
    fit_path, param_scales=PARAM_SCALES,
    data=data, trial_mask=trial_mask,
    model_factory=factory, redo_fits=REDO_FITS, best_of=BEST_OF,
)
print(f"{n_subjects} subjects")

In [ ]:
# Pre-cache scales for standard tier
_pre_cache_scales = dict(
    break_drift_scale=DEFAULT_BREAK_DRIFT_SCALE,
    break_mcf_scale=DEFAULT_BREAK_MCF_SCALE,
    reminder_start_drift_scale=DEFAULT_REMINDER_START_DRIFT_SCALE,
    reminder_drift_scale=DEFAULT_REMINDER_DRIFT_SCALE,
)

std_prep = prepare_sweep(
    params, paradigm, factory, cache_after="reminder",
    **_pre_cache_scales,
)
print(f"Standard tier cached: {n_subjects} subjects (list_length={paradigm.list_length})")

## Sweeping interference MCF learning rate

The first encoding-phase intensifier is encoding strength in the context-to-item association matrix (M\_CF).
During film encoding, the MCF learning rate follows the fitted primacy gradient — high at position 1, decaying toward 1.0 by position 16.
During the interference phase, each competitor's MCF learning rate is a scale factor times what the primacy gradient would assign at that position.
At scale = 0, competitors receive no M\_CF encoding; at scale = 1.0, they are encoded exactly as additional study items; at scale > 1, they are encoded more strongly than normal.

The sweep ranges from 0 to 3.0 in 10 steps.
Each subject uses their own fitted `encoding_drift_rate` during the interference phase, keeping context proximity constant across the sweep.

In [ ]:
mcf_scale_values = np.linspace(0, 3, 10)
mcf_recalls_4d, rng = run_sweep(
    std_prep, rng,
    interference_mcf_scale=mcf_scale_values,
    filler_drift_scale=DEFAULT_FILLER_DRIFT_SCALE,
    filler_mcf_scale=DEFAULT_FILLER_MCF_SCALE,
)

mcf_spcs, mcf_stats = [], []
for i in range(len(mcf_scale_values)):
    recalls = mcf_recalls_4d[i].reshape(-1, mcf_recalls_4d.shape[-1])
    recalls = standard_remap(recalls, paradigm)
    mcf_spcs.append(fixed_pres_spc(recalls, STD_PRESENTED))
    mcf_stats.append(film_recalled_stats(recalls, paradigm, n_subjects))

print(f"MCF scale sweep done: {mcf_scale_values.round(2)}")

In [ ]:
labels = ["{:.2f}".format(v) for v in mcf_scale_values]
colors = light_to_dark_colors(len(mcf_spcs))
means, ci_lo, ci_hi = zip(*mcf_stats)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_interference_spc(
    mcf_spcs, labels, n_film=N_FILM, n_break=N_BREAK_SHOWN,
    n_presented=STD_PRESENTED, color_cycle=colors, axis=axes[0],
)
if SHOW_FILLERS_IN_SPC and N_FILLER > 0:
    add_filler_boundary(
        axes[0], N_FILM, N_INTERFERENCE, N_FILLER, STD_PRESENTED,
        n_break=N_BREAK_SHOWN, show_fillers=True,
    )
axes[0].set_title("SPC by Interference MCF Scale", fontsize=16, pad=35)

plot_summary_dv(
    mcf_scale_values, means, ci_lo, ci_hi,
    xlabel="Interference MCF Scale", axis=axes[1],
)
axes[1].set_title("Film Items Recalled", fontsize=16, pad=35)


The left panel shows how the serial position curve changes as MCF scale increases. At scale = 0, competitors receive no encoding and are never recalled — the SPC matches the film-only baseline. As the scale grows, interference items accumulate retrieval support and the Luce choice rule redistributes recall probability away from film items, particularly those at late study positions whose recency advantage is directly contested.

The right panel condenses this into a single dependent variable: mean film items recalled with 95% CI. The monotonic decrease confirms that stronger M\_CF encoding of competitors produces more interference — the first of our three intensifiers.

## Sweeping interference encoding drift rate

The second intensifier is context proximity between film items and competitors.
The `encoding_drift_rate` parameter controls how much temporal context updates when each item is encoded.
During the film phase, this is the subject's fitted value.
During interference, we scale it by a factor: at scale < 1.0 drift is slower (competitors stay closer to film context); at scale = 1.0 context drifts at the same rate as during film encoding; at scale > 1.0 drift is faster (competitors are pushed further away).

We expect a dose-response curve: low scales produce maximal interference, and increasing scale progressively reduces it, approaching a ceiling where competitors are so contextually distant they barely compete.
We fix the interference MCF scale at 1.0 and sweep the drift scale from 0.2 to 1.6.

In [ ]:
drift_scale_values = np.linspace(0.2, 1.6, 10)
drift_recalls_4d, rng = run_sweep(
    std_prep, rng,
    interference_drift_scale=drift_scale_values,
    filler_drift_scale=DEFAULT_FILLER_DRIFT_SCALE,
    filler_mcf_scale=DEFAULT_FILLER_MCF_SCALE,
)

drift_spcs, drift_stats = [], []
for i in range(len(drift_scale_values)):
    recalls = drift_recalls_4d[i].reshape(-1, drift_recalls_4d.shape[-1])
    recalls = standard_remap(recalls, paradigm)
    drift_spcs.append(fixed_pres_spc(recalls, STD_PRESENTED))
    drift_stats.append(film_recalled_stats(recalls, paradigm, n_subjects))

print(f"Drift scale sweep done: {drift_scale_values.round(2)}")

In [ ]:
labels = ["{:.2f}".format(v) for v in drift_scale_values]
colors = light_to_dark_colors(len(drift_spcs))
means, ci_lo, ci_hi = zip(*drift_stats)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_interference_spc(
    drift_spcs, labels, n_film=N_FILM, n_break=N_BREAK_SHOWN,
    n_presented=STD_PRESENTED, color_cycle=colors, axis=axes[0],
)
if SHOW_FILLERS_IN_SPC and N_FILLER > 0:
    add_filler_boundary(
        axes[0], N_FILM, N_INTERFERENCE, N_FILLER, STD_PRESENTED,
        n_break=N_BREAK_SHOWN, show_fillers=True,
    )
axes[0].set_title("SPC by Interference Drift Scale", fontsize=16, pad=35)

plot_summary_dv(
    drift_scale_values, means, ci_lo, ci_hi,
    xlabel="Interference Drift Scale", axis=axes[1],
)
axes[1].set_title("Film Items Recalled", fontsize=16, pad=35)


Low drift scales produce the most interference: competitors encoded in film-like context compete strongly at retrieval, suppressing film recall. As the scale increases, competitors are pushed into increasingly distant context states, reducing their activation during retrieval and allowing more film items to be recalled. The curve approaches a ceiling at high scales — beyond a certain point, further increases produce negligible additional relief because the competitors are already contextually irrelevant.

### Context trajectory during interference

To visualize how context drifts during the interference phase, we track the cosine similarity between the current context and the pre-interference context after each encoding step, at several drift scale values.

In [ ]:
from jaxcmr.selective_interference import configure_rates


def _track_context(model, items, reference):
    """Track context similarity to reference during interference encoding."""
    ref_normed = normalize_magnitude(reference)

    def step(m, i):
        m = m.experience_interference(items[i])
        sim = jnp.dot(normalize_magnitude(m.context.state), ref_normed)
        return m, sim

    _, trajectory = lax.scan(step, model, jnp.arange(items.shape[0]))
    return trajectory


traj_drift_values = np.array([0.2, 0.5, 0.8, 1.0, 1.4, 1.8])
fig, ax = plt.subplots(figsize=(8, 5))
colors = plt.colormaps["Greys"](np.linspace(0.25, 0.95, len(traj_drift_values)))

for ds, color in zip(traj_drift_values, colors):
    ready = configure_rates(
        std_prep.models,
        interference_drift_scale=float(ds),
    )
    traj = jit(vmap(
        lambda m: _track_context(m, paradigm.interference_items, m.context.state)
    ))(ready)
    mean_traj = np.asarray(jnp.mean(traj[:, :N_INTERFERENCE], axis=0))
    ax.plot(
        np.arange(1, N_INTERFERENCE + 1), mean_traj, "o-", color=color,
        label=f"{ds:.1f}", markersize=4, linewidth=1.5,
    )

ax.set_xlabel("Interference Encoding Step", fontsize=16)
ax.set_ylabel("Similarity to Pre-Interference Context", fontsize=16)
ax.set_title("Context Trajectory During Interference", fontsize=16)
ax.tick_params(labelsize=14)
for loc in ("top", "right"):
    ax.spines[loc].set_visible(False)
ax.yaxis.grid(True, linestyle="--", linewidth=0.5, alpha=0.3)
ax.legend(title="Drift Scale", loc="upper right", fontsize=11)
fig.tight_layout()
plt.show()

## Sweeping competitor count

The third intensifier is the sheer number of competitors.
With interference drift and MCF scales both fixed at 1.0 (interference items encoded identically to additional study items), adding more competitors should reduce film recall — but with diminishing returns, because later competitors drift further from film context and therefore compete less effectively.

In [ ]:
m_values = np.arange(10, 33, 2, dtype=int)
print(f"Competitor count sweep values: {m_values}")

# Interference-extended paradigm
ext_paradigm = Paradigm(
    n_film=N_FILM, n_break=N_BREAK,
    n_interference=N_INTERFERENCE, n_interference_max=N_INTERFERENCE_MAX,
    n_filler=N_FILLER,
    experiment_count=EXPERIMENT_COUNT,
)
COMP_PRESENTED = compute_n_presented(
    ext_paradigm, n_interference=N_INTERFERENCE_MAX,
    show_break=SHOW_BREAK_IN_SPC, show_fillers=SHOW_FILLERS_IN_SPC,
)

ext_prep = prepare_sweep(
    params, ext_paradigm, factory, cache_after="reminder",
    tier="interference_extended",
    **_pre_cache_scales,
)

comp_spcs, comp_stats = [], []

for m in m_values:
    interf_items = make_extended_interference(ext_paradigm, int(m))
    # item_args = (interf_items, filler_items, max_recall); override interf_items
    custom_args = (interf_items,) + ext_prep.item_args[1:]
    rngs_2d, rng = sweep_rngs(rng, n_subjects, EXPERIMENT_COUNT)
    recalls_3d = ext_prep.batched(ext_prep.models, rngs_2d, *custom_args)
    recalls = recalls_3d.reshape(-1, recalls_3d.shape[-1])
    recalls = interference_extended_remap(
        recalls, ext_paradigm, n_interference=N_INTERFERENCE_MAX,
    )
    comp_spcs.append(fixed_pres_spc(recalls, COMP_PRESENTED))
    comp_stats.append(film_recalled_stats(recalls, ext_paradigm, n_subjects))

print("Competitor count sweep done")

In [ ]:
labels = [f"M={m}" for m in m_values]
colors = light_to_dark_colors(len(comp_spcs))
means, ci_lo, ci_hi = zip(*comp_stats)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

n_presented_comp = [
    N_FILM + N_BREAK_SHOWN + int(m)
    + (N_FILLER if SHOW_FILLERS_IN_SPC else 0)
    for m in m_values
]
plot_interference_spc(
    comp_spcs, labels, N_FILM,
    n_break=N_BREAK_SHOWN, n_presented=n_presented_comp,
    color_cycle=colors, axis=axes[0],
)
add_filler_boundary(
    axes[0], N_FILM, N_INTERFERENCE_MAX, N_FILLER, COMP_PRESENTED,
    n_break=N_BREAK_SHOWN,
)
axes[0].set_title("SPC by Competitor Count", fontsize=16, pad=20)

plot_summary_dv(m_values, means, ci_lo, ci_hi,
                xlabel="Number of Competitors (M)", axis=axes[1])
axes[1].set_title("Film Items Recalled", fontsize=16, pad=20)
plt.tight_layout()
plt.show()

More competitors produce more interference, but with diminishing returns as predicted. The first few competitors — those encoded in context most similar to the film items — have the strongest suppressive effect. Later competitors are encoded after more contextual drift and contribute progressively less interference. The SPC shows the interference effect concentrated at late film positions: recency is eroded first, while primacy is relatively protected by the strong M\_CF encoding at early positions.

This diminishing-returns pattern has an important empirical implication: the *timing* of interference (context proximity) matters more than the *amount* (raw count). A small number of competitors encoded in highly similar context can produce more interference than many competitors encoded after substantial drift.

## Summary

Three manipulations intensify interference from competing events, each operating through a distinct mechanism in the retrieved-context framework:

1. **Interference MCF encoding strength** increases retrieval support for competitors, redistributing probability mass away from film items under the Luce choice rule. The effect is monotonic.

2. **Interference encoding drift rate** controls context proximity. Low scales keep competitors in film-adjacent context; high scales push them into contextually distant states. The effect follows a dose-response curve with a ceiling.

3. **Competitor count** adds more entries into the association matrices, but with diminishing returns: later competitors accumulate more contextual drift. The timing of interference matters more than the raw amount.

**Postdictions**: More engaging interference tasks produce stronger effects (MCF encoding); simply increasing task duration shows diminishing returns (Holmes et al. 2009; James et al. 2015).

**Predictions**: Interference is primarily driven by MCF encoding strength and context proximity, not count alone. Sequential encoding naturally produces a recency gradient — late-film items are disproportionately suppressed because they share the most temporal context with competitors.

In [ ]:
print("Default parameters (non-swept variables):")
print("  Interference MCF scale:       1.0")
print("  Interference drift scale:     1.0")
print(f"  Break count:                  {N_BREAK}")
print(f"  Break drift scale:            {DEFAULT_BREAK_DRIFT_SCALE}")
print(f"  Break MCF scale:              {DEFAULT_BREAK_MCF_SCALE}")
print(f"  Reminder start drift scale:   {DEFAULT_REMINDER_START_DRIFT_SCALE}")
print(f"  Reminder drift scale:         {DEFAULT_REMINDER_DRIFT_SCALE}")
print(f"  Filler drift scale:           {DEFAULT_FILLER_DRIFT_SCALE}")
print(f"  Filler MCF scale:             {DEFAULT_FILLER_MCF_SCALE}")
print(f"  Competitor count (M):         {N_INTERFERENCE}")
print(f"  Filler count:                 {N_FILLER}")
print(f"  Max competitors (sweep):      {N_INTERFERENCE_MAX}")
print(f"  Subjects:                     {n_subjects}")
print(f"  Replications per sweep:       {EXPERIMENT_COUNT}")